In [10]:
import sqlite3
import pandas as pd

cdf = pd.read_csv("temp_flights.csv")

conn = sqlite3.connect("airspace.db")
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

cursor.execute("DROP TABLE IF EXISTS telemetry_logs;")
cursor.execute("DROP TABLE IF EXISTS flights;")
cursor.execute("DROP TABLE IF EXISTS aircraft;")

cursor.execute(
    """
    CREATE TABLE IF NOT EXISTS aircraft (
    icao24 TEXT PRIMARY KEY CHECK(length(icao24)=6),
    origin_country TEXT
    );
    """
)

cursor.execute(
    """
    CREATE TABLE IF NOT EXISTS flights (
    flight_id INTEGER PRIMARY KEY AUTOINCREMENT,
    icao24 TEXT,
    callsign TEXT,
    airline_code TEXT,
    FOREIGN KEY (icao24) REFERENCES aircraft(icao24)
    );
    """
)

cursor.execute(
    """
    CREATE TABLE IF NOT EXISTS telemetry_logs (
    log_id INTEGER PRIMARY KEY AUTOINCREMENT,
    flight_id INTEGER,
    latitude REAL,
    longitude REAL,
    baro_alt REAL,
    velocity_knots REAL,
    vertical_rate_fpm REAL,
    FOREIGN KEY (flight_id) REFERENCES flights(flight_id)
    );
    """
)

for row in cdf.itertuples(index=False):
    cursor.execute("INSERT OR IGNORE INTO aircraft (icao24, origin_country) VALUES (?, ?)", (row.icao24, row.origin_country))

    cursor.execute("INSERT OR IGNORE INTO flights (icao24, callsign, airline_code) VALUES (?, ?, ?)", 
                   (row.icao24, row.callsign, row.airline_code))
    current_flight_id = cursor.lastrowid

    cursor.execute("INSERT OR IGNORE INTO telemetry_logs (flight_id, latitude, longitude, baro_alt, velocity_knots, vertical_rate_fpm) VALUES (?, ?, ?, ?, ?, ?)", 
                   (current_flight_id, 
                    row.latitude, 
                    row.longitude, 
                    row.baro_alt, 
                    row.velocity_knots, 
                    row.vertical_rate_fpm))

conn.commit()